In [60]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
from dotenv import load_dotenv
import requests
import sqlite3
from pathlib import Path
import logging

load_dotenv()  # carga las variables del archivo .env


True

In [61]:


AEMET_API_KEY = os.getenv("AEMET_API_KEY")

# Fechas de descarga (ajusta según necesites)
FECHA_INICIO = "2023-01-01"
FECHA_FIN   = datetime.now() - timedelta(days=10)

print(f"Descargando datos desde {FECHA_INICIO} hasta {FECHA_FIN.strftime('%Y-%m-%d')}...")
print("API Key cargada:", "✅" if AEMET_API_KEY else "❌ No encontrada")


Descargando datos desde 2023-01-01 hasta 2026-04-23...
API Key cargada: ✅


In [62]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────

DB_PATH         = "ree_generacion.db"   # fichero SQLite local
TABLE_NAME      = "generacion"
DIAS_POR_TRAMO  = 30                    # con time_trunc=day, 30 días por tramo es seguro
SLEEP_ENTRE_REQ = 1.0                   # segundos entre peticiones (respetar API)

# Granularidad: "hour" | "day"
# "hour" → más datos pero la API REE falla con frecuencia en histórico
# "day"  → estable para histórico de años, suficiente para ML
TIME_TRUNC      = "day"

# Headers requeridos por la API de REE (sin ellos devuelve 400)
REE_HEADERS = {
    "Accept"      : "application/json",
    "Content-Type": "application/json",
    "Host"        : "apidatos.ree.es",
}

REE_URL = "https://apidatos.ree.es/es/datos/generacion/estructura-generacion"

RENOVABLES = {
    "Eólica", "Solar fotovoltaica", "Solar térmica",
    "Hidráulica", "Hidroeólica", "Otras renovables", "Residuos renovables"
}

In [63]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

In [64]:
# ─────────────────────────────────────────────
# CAPA DE BASE DE DATOS
# ─────────────────────────────────────────────

def _get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL")   # escrituras más rápidas
    return conn


def _init_db():
    """Crea la tabla si no existe."""
    with _get_conn() as conn:
        conn.execute(f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
                datetime    TEXT NOT NULL,
                tecnologia  TEXT NOT NULL,
                tipo        TEXT,
                valor_MWh   REAL,
                porcentaje  REAL,
                PRIMARY KEY (datetime, tecnologia)
            )
        """)
        # Índice para acelerar consultas por fecha
        conn.execute(f"""
            CREATE INDEX IF NOT EXISTS idx_datetime
            ON {TABLE_NAME} (datetime)
        """)
    log.info(f"BD lista: {DB_PATH}")
    
def _fechas_en_bd() -> tuple[str | None, str | None]:
    """Devuelve (fecha_min, fecha_max) de lo que hay en la BD, o (None, None)."""
    with _get_conn() as conn:
        row = conn.execute(
            f"SELECT MIN(datetime), MAX(datetime) FROM {TABLE_NAME}"
        ).fetchone()
    return row if row else (None, None)


def _guardar_df(df: pd.DataFrame):
    """Inserta filas nuevas, ignora duplicados (INSERT OR IGNORE)."""
    if df.empty:
        return
    with _get_conn() as conn:
        df.to_sql("_tmp", conn, if_exists="replace", index=False)
        conn.execute(f"""
            INSERT OR IGNORE INTO {TABLE_NAME}
                (datetime, tecnologia, tipo, valor_MWh, porcentaje)
            SELECT datetime, tecnologia, tipo, valor_MWh, porcentaje
            FROM _tmp
        """)
        conn.execute("DROP TABLE _tmp")
    log.info(f"  ✅ Guardados {len(df)} registros")


def _leer_bd(fecha_inicio: str, fecha_fin: str,
             solo_renovables: bool = True) -> pd.DataFrame:
    """Lee de la BD el rango indicado."""
    query = f"""
        SELECT * FROM {TABLE_NAME}
        WHERE datetime >= ? AND datetime <= ?
        {"AND tecnologia IN (" + ",".join("?"*len(RENOVABLES)) + ")" if solo_renovables else ""}
        ORDER BY datetime
    """
    params = [fecha_inicio, fecha_fin]
    if solo_renovables:
        params += list(RENOVABLES)

    with _get_conn() as conn:
        df = pd.read_sql_query(query, conn, params=params)

    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    return df


In [65]:
# ─────────────────────────────────────────────
# CAPA DE API
# ─────────────────────────────────────────────

def _consultar_api(fecha_inicio: str, fecha_fin: str) -> pd.DataFrame:
    """
    Hace UNA petición a la API de REE para el rango dado.
    Devuelve DataFrame con los registros de renovables, o DataFrame vacío si falla.

    IMPORTANTE: la URL se construye a mano (sin params={}) porque requests codifica
    los ':' de las fechas como %3A y la API de REE devuelve 400 con esa codificación.
    """
    # URL construida a mano para evitar que requests codifique los ':' de las fechas
    url = (
        f"{REE_URL}"
        f"?start_date={fecha_inicio}T00:00"
        f"&end_date={fecha_fin}T23:59"
        f"&time_trunc={TIME_TRUNC}"
        f"&geo_trunc=electric_system"
        f"&geo_limit=peninsular"
        f"&geo_ids=8741"
    )

    try:
        r = requests.get(url, headers=REE_HEADERS, timeout=30)
    except requests.exceptions.RequestException as e:
        log.warning(f"  ⚠️  Error de red: {e}")
        return pd.DataFrame()

    if r.status_code != 200:
        log.warning(f"  ⚠️  HTTP {r.status_code} para {fecha_inicio} → {fecha_fin}: {r.text[:120]}")
        return pd.DataFrame()

    tecnologias = r.json().get("included", [])
    registros = []

    for tec in tecnologias:
        nombre = tec.get("attributes", {}).get("title", "")
        tipo   = tec.get("attributes", {}).get("type", "")

        if nombre not in RENOVABLES:
            continue

        for val in tec.get("attributes", {}).get("values", []):
            registros.append({
                "datetime"  : val.get("datetime"),
                "tecnologia": nombre,
                "tipo"      : tipo,
                "valor_MWh" : val.get("value"),
                "porcentaje": val.get("percentage"),
            })

    if not registros:
        log.warning(f"  ⚠️  Sin registros para {fecha_inicio} → {fecha_fin}")
        return pd.DataFrame()

    df = pd.DataFrame(registros)
    # Normalizar datetime a UTC con zona horaria para consistencia
    df["datetime"] = (
        pd.to_datetime(df["datetime"], utc=True)
          .dt.strftime("%Y-%m-%dT%H:%M:%SZ")   # guardar como texto ISO en SQLite
    )
    return df


# ─────────────────────────────────────────────
# FUNCIÓN PRINCIPAL 1: GENERAR HISTÓRICO
# ─────────────────────────────────────────────

def generar_historico(fecha_inicio: str, fecha_fin: str,
                      dias_por_tramo: int = DIAS_POR_TRAMO,
                      forzar: bool = False):
    """
    Descarga el histórico completo en tramos cortos y lo guarda en SQLite.

    Args:
        fecha_inicio  : "YYYY-MM-DD"
        fecha_fin     : "YYYY-MM-DD"
        dias_por_tramo: días por petición (7 recomendado para granularidad horaria)
        forzar        : si True, re-descarga aunque ya exista en BD

    Ejemplo:
        generar_historico("2020-01-01", "2024-12-31")
    """
    _init_db()

    inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
    fin    = datetime.strptime(fecha_fin,    "%Y-%m-%d")

    # Calcular tramos pendientes
    tramos = []
    actual = inicio
    while actual <= fin:
        tramo_fin = min(actual + timedelta(days=dias_por_tramo - 1), fin)
        tramos.append((actual.strftime("%Y-%m-%d"), tramo_fin.strftime("%Y-%m-%d")))
        actual = tramo_fin + timedelta(days=1)

    total = len(tramos)
    log.info(f"📅 Histórico {fecha_inicio} → {fecha_fin} | {total} tramos de {dias_por_tramo} días")

    # Si ya hay datos en BD y no forzamos, saltar los tramos ya descargados
    fecha_min_bd, fecha_max_bd = _fechas_en_bd()

    descargados = 0
    omitidos    = 0

    for i, (t_ini, t_fin) in enumerate(tramos, 1):
        # Comprobar si este tramo ya está en BD
        if not forzar and fecha_max_bd:
            t_ini_dt = datetime.strptime(t_ini, "%Y-%m-%d")
            t_max_bd = datetime.strptime(fecha_max_bd[:10], "%Y-%m-%d")
            if t_ini_dt <= t_max_bd:
                omitidos += 1
                continue

        log.info(f"  [{i:03d}/{total}] Descargando {t_ini} → {t_fin}...")
        df = _consultar_api(t_ini, t_fin)

        if not df.empty:
            _guardar_df(df)
            descargados += 1
        else:
            log.warning(f"  [{i:03d}/{total}] Sin datos — tramo omitido")

        time.sleep(SLEEP_ENTRE_REQ)

    log.info(f"\n✅ Histórico completado | descargados: {descargados} tramos | omitidos (ya en BD): {omitidos}")
    _mostrar_resumen_bd()


# ─────────────────────────────────────────────
# FUNCIÓN PRINCIPAL 2: OBTENER DATOS (con caché)
# ─────────────────────────────────────────────

def obtener_datos(fecha_inicio: str, fecha_fin: str,
                  solo_renovables: bool = True) -> pd.DataFrame:
    """
    Devuelve datos del período solicitado.
    - Si están en la BD → los lee directamente (sin llamar a la API)
    - Si hay un hueco al final (datos recientes) → los descarga y cachea

    Args:
        fecha_inicio   : "YYYY-MM-DD"
        fecha_fin      : "YYYY-MM-DD"
        solo_renovables: si True filtra solo tecnologías renovables

    Returns:
        DataFrame con columnas: datetime, tecnologia, tipo, valor_MWh, porcentaje

    Ejemplo:
        df = obtener_datos("2023-01-01", "2024-12-31")
    """
    _init_db()

    fecha_min_bd, fecha_max_bd = _fechas_en_bd()

    fin_dt = datetime.strptime(fecha_fin, "%Y-%m-%d")

    # ¿Hay datos recientes que faltan en la BD?
    if fecha_max_bd:
        ultimo_en_bd = datetime.strptime(fecha_max_bd[:10], "%Y-%m-%d")
        if ultimo_en_bd < fin_dt:
            # Solo descargar el período que falta
            nuevo_inicio = (ultimo_en_bd + timedelta(days=1)).strftime("%Y-%m-%d")
            log.info(f"🔄 Actualizando BD: {nuevo_inicio} → {fecha_fin}")
            _descargar_rango_incremental(nuevo_inicio, fecha_fin)
        else:
            log.info("✅ BD al día, sin necesidad de consultar la API")
    else:
        # BD vacía → descargar todo
        log.info(f"📭 BD vacía. Descargando {fecha_inicio} → {fecha_fin}")
        _descargar_rango_incremental(fecha_inicio, fecha_fin)

    # Leer de la BD y devolver
    df = _leer_bd(fecha_inicio, fecha_fin, solo_renovables=solo_renovables)
    log.info(f"📊 Retornando {len(df)} registros desde la BD")
    return df


def _descargar_rango_incremental(fecha_inicio: str, fecha_fin: str):
    """Descarga un rango en tramos y lo guarda. Uso interno."""
    inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
    fin    = datetime.strptime(fecha_fin,    "%Y-%m-%d")
    actual = inicio

    while actual <= fin:
        tramo_fin = min(actual + timedelta(days=DIAS_POR_TRAMO - 1), fin)
        t_ini_str = actual.strftime("%Y-%m-%d")
        t_fin_str = tramo_fin.strftime("%Y-%m-%d")

        log.info(f"  → Descargando {t_ini_str} → {t_fin_str}")
        df = _consultar_api(t_ini_str, t_fin_str)

        if not df.empty:
            _guardar_df(df)

        actual = tramo_fin + timedelta(days=1)
        time.sleep(SLEEP_ENTRE_REQ)


# ─────────────────────────────────────────────
# UTILIDADES
# ─────────────────────────────────────────────

def _mostrar_resumen_bd():
    fecha_min, fecha_max = _fechas_en_bd()
    with _get_conn() as conn:
        n_filas = conn.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}").fetchone()[0]
        tecns   = conn.execute(
            f"SELECT tecnologia, COUNT(*) as n FROM {TABLE_NAME} GROUP BY tecnologia"
        ).fetchall()

    log.info(f"\n📦 Estado de la BD ({DB_PATH}):")
    log.info(f"   Rango   : {fecha_min} → {fecha_max}")
    log.info(f"   Total   : {n_filas:,} filas")
    log.info(f"   Por tecnología:")
    for tec, n in tecns:
        log.info(f"     {tec:<30} {n:>8,} filas")


def exportar_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte el DataFrame long (una fila por tecnología/hora)
    a formato wide (una columna por tecnología) listo para ML.
    """
    df_pivot = df.pivot_table(
        index="datetime",
        columns="tecnologia",
        values="valor_MWh",
        aggfunc="sum"
    ).reset_index()

    df_pivot.columns.name = None

    # Features temporales útiles para ML
    df_pivot["hora"]          = df_pivot["datetime"].dt.hour
    df_pivot["dia_semana"]    = df_pivot["datetime"].dt.dayofweek
    df_pivot["mes"]           = df_pivot["datetime"].dt.month
    df_pivot["es_fin_semana"] = df_pivot["dia_semana"].isin([5, 6]).astype(int)
    df_pivot["trimestre"]     = df_pivot["datetime"].dt.quarter

    return df_pivot

In [67]:
    # ── Paso 1: Generar histórico (solo la primera vez, o cuando quieras ampliar)
    # Descarga 2022 y 2023 en tramos de 7 días → ~104 peticiones
ayer = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
generar_historico("2022-01-01", ayer)

    # ── Paso 2: Uso cotidiano — obtener datos actualizados
    # Solo descarga de la API lo que no está ya en la BD
df = obtener_datos("2022-01-01", ayer)

print(f"\nDataFrame: {df.shape}")
print(df.head())

    # ── Paso 3: Convertir a formato wide para ML
df_ml = exportar_pivot(df)
print(f"\nDataFrame para ML: {df_ml.shape}")
print(df_ml.columns.tolist())

    # ── Opcional: exportar a CSV
df_ml.to_csv("dataset_ree_para_ml.csv", index=False)
print("\n💾 Guardado: dataset_ree_para_ml.csv")

20:44:20  INFO  BD lista: ree_generacion.db
20:44:20  INFO  📅 Histórico 2022-01-01 → 2026-05-02 | 53 tramos de 30 días
20:44:20  INFO    [026/53] Descargando 2024-01-21 → 2024-02-19...
20:44:21  INFO    ✅ Guardados 180 registros
20:44:22  INFO    [027/53] Descargando 2024-02-20 → 2024-03-20...
20:44:23  INFO    ✅ Guardados 180 registros
20:44:24  INFO    [028/53] Descargando 2024-03-21 → 2024-04-19...
20:44:26  INFO    ✅ Guardados 191 registros
20:44:27  INFO    [029/53] Descargando 2024-04-20 → 2024-05-19...
20:44:28  INFO    ✅ Guardados 180 registros
20:44:29  INFO    [030/53] Descargando 2024-05-20 → 2024-06-18...
20:44:31  INFO    ✅ Guardados 180 registros
20:44:32  INFO    [031/53] Descargando 2024-06-19 → 2024-07-18...
20:44:33  INFO    ✅ Guardados 180 registros
20:44:34  INFO    [032/53] Descargando 2024-07-19 → 2024-08-17...
20:44:36  INFO    ✅ Guardados 180 registros
20:44:37  INFO    [033/53] Descargando 2024-08-18 → 2024-09-16...
20:44:38  INFO    ✅ Guardados 180 registros
2


DataFrame: (9372, 5)
                   datetime          tecnologia       tipo   valor_MWh  \
0 2022-01-01 23:00:00+00:00          Hidráulica  Renovable   52663.908   
1 2022-01-01 23:00:00+00:00              Eólica  Renovable  103224.219   
2 2022-01-01 23:00:00+00:00  Solar fotovoltaica  Renovable   42032.416   
3 2022-01-01 23:00:00+00:00       Solar térmica  Renovable    4509.884   
4 2022-01-01 23:00:00+00:00    Otras renovables  Renovable   14173.180   

   porcentaje  
0    0.097975  
1    0.192036  
2    0.078196  
3    0.008390  
4    0.026367  

DataFrame para ML: (1562, 12)
['datetime', 'Eólica', 'Hidráulica', 'Otras renovables', 'Residuos renovables', 'Solar fotovoltaica', 'Solar térmica', 'hora', 'dia_semana', 'mes', 'es_fin_semana', 'trimestre']

💾 Guardado: dataset_ree_para_ml.csv


In [68]:
df_ree = pd.read_csv("dataset_ree_para_ml.csv")
df_ree

,datetime,Eólica,Hidráulica,Otras renovables,Residuos renovables,Solar fotovoltaica,Solar térmica,hora,dia_semana,mes,es_fin_semana,trimestre
0,2022-01-01 23:00:00+00:00,103224.219,52663.908,14173.180,2387.7820,42032.416,4509.884,23,5,1,1,1
1,2022-01-02 23:00:00+00:00,151458.364,53068.508,14249.328,2388.4350,46275.789,6250.758,23,6,1,1,1
2,2022-01-03 23:00:00+00:00,302569.215,51497.125,14951.210,2361.4625,26961.168,1141.307,23,0,1,0,1
3,2022-01-04 23:00:00+00:00,288109.621,57195.362,14758.413,2387.6545,33969.307,1970.000,23,1,1,0,1
4,2022-01-05 23:00:00+00:00,196473.458,60819.354,14508.809,2363.1080,51808.514,6858.916,23,2,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1557,2026-04-27 22:00:00+00:00,122253.200,78588.000,9271.700,1478.2000,170050.245,7170.555,22,0,4,0,2
1558,2026-04-28 22:00:00+00:00,124158.500,94584.600,8773.700,1519.3000,140152.303,3302.497,22,1,4,0,2
1559,2026-04-29 22:00:00+00:00,48040.500,101710.500,9241.000,1535.7500,199773.467,12953.733,22,2,4,0,2
1560,2026-04-30 22:00:00+00:00,66377.300,83383.300,10122.800,1533.6500,147621.343,16010.157,22,3,4,0,2
